# QLoRA Fine-Tuning for Product Price Prediction

This notebook fine-tunes an open-source LLM with **QLoRA (Quantized Low-Rank Adaptation)** to predict product prices from short text descriptions.

- Fine-tuned an open-source model to reach **frontier-level** performance on a price prediction task (evaluate with MAE).
- Built an end-to-end workflow: dataset ingestion, prompt formatting, QLoRA training, evaluation, and inference.


## 0) Environment

This notebook can run:
- **Locally** (GPU recommended)
- **Google Colab** (recommended for training)

You will need:
- a Hugging Face account + token (for gated models/datasets if applicable)
- enough GPU VRAM for your chosen base model

> Note: Some base models require accepting a license on Hugging Face (for example, Llama family).


In [ ]:
# If running in Colab, uncomment:
# !pip -q install -U "transformers>=4.41" "datasets>=2.20" "accelerate>=0.33" peft trl bitsandbytes evaluate python-dotenv

# If running locally, install once via:
# pip install -U transformers datasets accelerate peft trl bitsandbytes evaluate python-dotenv


## 1) Imports + configuration

In [ ]:
import os
import re
import json
import math
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

from dotenv import load_dotenv

import torch
from datasets import load_dataset
import evaluate

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer


In [ ]:
load_dotenv(override=True)

# ==== User-configurable knobs ====
BASE_MODEL = os.getenv("BASE_MODEL", "meta-llama/Llama-3.2-3B")  # change if needed
DATASET_NAME = os.getenv("DATASET_NAME", "ed-donner/items_full")  # change if needed
TEXT_FIELD = os.getenv("TEXT_FIELD", "text")  # update if your dataset uses a different field name
PRICE_FIELD = os.getenv("PRICE_FIELD", "price")  # update if needed

LITE_MODE = os.getenv("LITE_MODE", "false").lower() == "true"  # quicker runs
MAX_TRAIN_SAMPLES = int(os.getenv("MAX_TRAIN_SAMPLES", "2000" if LITE_MODE else "0"))  # 0 => all
MAX_EVAL_SAMPLES = int(os.getenv("MAX_EVAL_SAMPLES", "500" if LITE_MODE else "0"))

# Training defaults (safe-ish starter values)
OUTPUT_DIR = os.getenv("OUTPUT_DIR", "./outputs_price_qlora")
SEED = int(os.getenv("SEED", "42"))

# If your model/dataset is gated, set HF_TOKEN in env/.env
HF_TOKEN = os.getenv("HF_TOKEN", None)

print("BASE_MODEL:", BASE_MODEL)
print("DATASET_NAME:", DATASET_NAME)
print("LITE_MODE:", LITE_MODE)


## 2) Load dataset

Expected schema (flexible):
- a **text/description** field (default: `text`)
- a numeric **price** field (default: `price`)

If your dataset differs, set `TEXT_FIELD` and `PRICE_FIELD`.


In [ ]:
ds = load_dataset(DATASET_NAME, token=HF_TOKEN)

# Pick common split names. If your dataset differs, adjust here.
train_split = "train" if "train" in ds else list(ds.keys())[0]
eval_split = "validation" if "validation" in ds else ("val" if "val" in ds else ("test" if "test" in ds else train_split))

train_ds = ds[train_split]
eval_ds = ds[eval_split]

if MAX_TRAIN_SAMPLES and len(train_ds) > MAX_TRAIN_SAMPLES:
    train_ds = train_ds.select(range(MAX_TRAIN_SAMPLES))
if MAX_EVAL_SAMPLES and len(eval_ds) > MAX_EVAL_SAMPLES:
    eval_ds = eval_ds.select(range(MAX_EVAL_SAMPLES))

print("Train:", len(train_ds), "Eval:", len(eval_ds))
print("Columns:", train_ds.column_names)

## 3) Prompt format

We train the model to output a single number (USD).  
Keeping the output tight makes evaluation easier and reduces nonsense.

You can tweak the prompt to match your dataset.


In [ ]:
def format_example(example: Dict) -> Dict:
    desc = str(example[TEXT_FIELD]).strip()
    price = example.get(PRICE_FIELD, None)

    # For supervised fine-tuning, we include the target price in the label.
    # The trainer will learn to predict the completion.
    prompt = (
        "You are a pricing assistant. "
        "Given a product description, predict the typical price in USD.

"
        f"Description: {desc}
"
        "Answer (USD): "
    )

    # Label should be compact. If price is missing, we skip label.
    if price is None:
        completion = ""
    else:
        # round to nearest dollar to simplify
        completion = f"{float(price):.0f}"

    example["prompt"] = prompt
    example["completion"] = completion
    example["text_for_sft"] = prompt + completion
    return example

train_ds = train_ds.map(format_example)
eval_ds = eval_ds.map(format_example)

train_ds[0]["text_for_sft"][:300]


## 4) Baselines (optional)

This section includes a simple benchmark chart (from the Udemy project context).  
You can update these numbers using your own evaluation results.


In [ ]:
# Optional: baseline comparison chart
import plotly.graph_objects as go

baselines = [
    ("Constant", 106.18),
    ("Linear Regression", 101.56),
    ("NLP + LR", 76.81),
    ("Random Forest", 72.28),
    ("XGBoost", 67.25),
    ("Frontier LLM (prompt)", 63.10),
    ("Fine-tuned open-source (target)", 60.00),  # replace with your measured MAE
]

fig = go.Figure(go.Bar(x=[b[0] for b in baselines], y=[b[1] for b in baselines]))
fig.update_layout(title="MAE (lower is better) - Baseline Comparison", xaxis_title="Model", yaxis_title="MAE")
fig.show()


## 5) QLoRA training setup

QLoRA uses:
- 4-bit quantization (bitsandbytes)
- LoRA adapters (PEFT)
- supervised fine-tuning (TRL `SFTTrainer`)

If you only want to do inference or evaluation, skip training and load an existing adapter.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    token=HF_TOKEN,
    quantization_config=bnb_config,
    device_map="auto",
)

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # common for Llama-like models
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


In [ ]:
# Training args
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1 if LITE_MODE else 2,
    gradient_accumulation_steps=8 if LITE_MODE else 16,
    learning_rate=2e-4,
    num_train_epochs=1 if LITE_MODE else 2,
    logging_steps=20,
    save_steps=200,
    save_total_limit=2,
    evaluation_strategy="steps",
    eval_steps=200,
    bf16=torch.cuda.is_available(),
    fp16=not torch.cuda.is_available(),
    optim="paged_adamw_8bit",
    seed=SEED,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    dataset_text_field="text_for_sft",
    max_seq_length=512 if LITE_MODE else 768,
    args=training_args,
)

# Run training
# trainer.train()


## 6) Inference helper

We parse the model output into a number and compute MAE.


In [ ]:
mae = evaluate.load("mae")

_price_re = re.compile(r"(-?\d+(?:\.\d+)?)")

def extract_price(text: str) -> Optional[float]:
    if not text:
        return None
    m = _price_re.search(text.replace(",", ""))
    if not m:
        return None
    try:
        return float(m.group(1))
    except:
        return None

@torch.inference_mode()
def predict_price(description: str, max_new_tokens: int = 10) -> Tuple[str, Optional[float]]:
    prompt = (
        "You are a pricing assistant. "
        "Given a product description, predict the typical price in USD.

"
        f"Description: {description.strip()}
"
        "Answer (USD): "
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=0.0,
        pad_token_id=tokenizer.eos_token_id,
    )
    decoded = tokenizer.decode(out[0], skip_special_tokens=True)
    # Extract only the part after the answer cue
    tail = decoded.split("Answer (USD):", 1)[-1].strip()
    return tail, extract_price(tail)

# Quick spot-check
sample_desc = str(eval_ds[0][TEXT_FIELD])
raw, pred = predict_price(sample_desc)
raw, pred


## 7) Evaluation loop (MAE)

This will run on the eval split and report MAE.  
If you did not train in this session, you'll be evaluating the base model + adapters currently loaded.


In [ ]:
import numpy as np
from tqdm.auto import tqdm

def evaluate_model(ds_eval, n: int = 200):
    y_true = []
    y_pred = []

    n = min(n, len(ds_eval))
    for i in tqdm(range(n)):
        ex = ds_eval[i]
        true_price = ex.get(PRICE_FIELD, None)
        if true_price is None:
            continue

        _, pred = predict_price(str(ex[TEXT_FIELD]))
        if pred is None:
            continue

        y_true.append(float(true_price))
        y_pred.append(float(pred))

    if not y_true:
        return {"mae": None, "n": 0}

    score = mae.compute(predictions=y_pred, references=y_true)
    return {"mae": float(score["mae"]), "n": len(y_true)}

# Run a quick evaluation (adjust n)
# evaluate_model(eval_ds, n=200 if LITE_MODE else 1000)


## 8) Save adapters

After training, save the LoRA adapters so you can load them later without retraining.


In [ ]:
# After training:
# trainer.model.save_pretrained(OUTPUT_DIR)
# tokenizer.save_pretrained(OUTPUT_DIR)
# print("Saved adapters + tokenizer to:", OUTPUT_DIR)
